# Atividade 6 — Matplotlib, melt e merge (revisão PI)

Junta as duas **Tabelas 5** (participação de homens e mulheres entre as pessoas desocupadas, 2012 T1 e 2026 T1), compara os anos e cruza com a **Tabela 1.1.1** (horas semanais de cuidados e afazeres domésticos, 2022), usando só os estados.

Arquivos na mesma pasta: `Tabela5-sem_emprego_2012.csv`, `Tabela5-sem_emprego_2026.csv` e `Tabela 1.1.1.xls`.

In [ ]:
# python -m pip install pandas matplotlib seaborn xlrd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def ler_csv(arquivo, **kw):
    """CSV do IBGE: separador ; e vírgula decimal. Tenta utf-8 e depois latin-1."""
    try:
        return pd.read_csv(arquivo, sep=";", decimal=",", encoding="utf-8", **kw)
    except UnicodeDecodeError:
        return pd.read_csv(arquivo, sep=";", decimal=",", encoding="latin-1", **kw)

## Abrir as duas Tabelas 5

In [ ]:
t2012 = ler_csv("Tabela5-sem_emprego_2012.csv")
t2026 = ler_csv("Tabela5-sem_emprego_2026.csv")

t2012.columns = ["Sigla", "Código", "Estado", "homens_2012", "mulheres_2012"]
t2026.columns = ["Sigla", "Código", "Estado", "homens_2026", "mulheres_2026"]
t2012.head()

## Juntar (`merge`) e comparar ano a ano

In [ ]:
comp = t2012.merge(t2026, on=["Sigla", "Código", "Estado"], how="inner")
comp["variacao_mulheres"] = (comp["mulheres_2026"] - comp["mulheres_2012"]).round(1)
comp.sort_values("variacao_mulheres", ascending=False)

In [ ]:
print("Média da participação feminina 2012:", round(comp["mulheres_2012"].mean(), 1))
print("Média da participação feminina 2026:", round(comp["mulheres_2026"].mean(), 1))
print("Estados em que a participação das mulheres aumentou:", (comp["variacao_mulheres"] > 0).sum())
print("Estados em que diminuiu:", (comp["variacao_mulheres"] < 0).sum())

## Tabela 1.1.1 (código da aula 2) — só os estados

In [ ]:
def ler_tabela_ibge(arquivo, nomes_colunas):
    bruto = pd.read_excel(arquivo, header=None)
    col0 = bruto.columns[0]
    inicio = bruto.index[bruto[col0] == "Brasil"][0]
    fim = bruto.index[bruto[col0] == "Distrito Federal"][0]
    df = bruto.loc[inicio:fim, bruto.columns[: len(nomes_colunas)]].copy()
    df.columns = nomes_colunas
    for c in nomes_colunas[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.reset_index(drop=True)

horas = ler_tabela_ibge(
    "Tabela 1.1.1.xls",
    ["Estado", "total", "total_branca", "total_preta_parda",
     "homem_branca", "homem_preta_parda", "mulher_branca", "mulher_preta_parda"],
)
# fica só com os estados (tira Brasil e as Grandes Regiões)
horas = horas[~horas["Estado"].isin(["Brasil", "Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"])]
horas["horas_mulheres"] = ((horas["mulher_branca"] + horas["mulher_preta_parda"]) / 2).round(1)
horas["horas_homens"] = ((horas["homem_branca"] + horas["homem_preta_parda"]) / 2).round(1)
horas = horas[["Estado", "horas_homens", "horas_mulheres"]]
print(len(horas), "estados")
horas.head()

In [ ]:
completa = comp.merge(horas, on="Estado", how="inner")
print(len(completa), "estados depois do merge")
completa.head()

In [ ]:
# Estados onde as mulheres dedicam mais horas à casa também têm mais mulheres desocupadas?
print("Correlação horas_mulheres x mulheres_2026:",
      round(completa["horas_mulheres"].corr(completa["mulheres_2026"]), 2))

## Gráfico usando o merge

In [ ]:
ordem = comp.sort_values("mulheres_2026", ascending=False)["Estado"]
longo = comp.melt(
    id_vars=["Sigla", "Código", "Estado"],
    value_vars=["mulheres_2012", "mulheres_2026"],
    var_name="Ano",
    value_name="Participacao_mulheres",
)
longo["Ano"] = longo["Ano"].map({"mulheres_2012": "2012 T1", "mulheres_2026": "2026 T1"})

fig, ax = plt.subplots(figsize=(10, 10))
sns.barplot(
    data=longo,
    y="Estado",
    x="Participacao_mulheres",
    hue="Ano",
    order=ordem,
    ax=ax,
)
ax.axvline(50, color="red", linestyle="--", linewidth=1)
ax.set_xlabel("Participação das mulheres entre as pessoas desocupadas (%)")
ax.set_ylabel("")
ax.set_title("Desocupação: participação feminina em 2012 T1 e 2026 T1")
ax.legend(title="Trimestre")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=completa, x="horas_mulheres", y="mulheres_2026", ax=ax)
for _, l in completa.iterrows():
    ax.annotate(l["Sigla"], (l["horas_mulheres"], l["mulheres_2026"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Horas semanais das mulheres em afazeres domésticos (2022)")
ax.set_ylabel("% de mulheres entre os desocupados (2026 T1)")
ax.set_title("Afazeres domésticos x desocupação feminina por estado")
fig.tight_layout()
plt.show()